In [0]:
from pyspark.sql import functions as f 
import datetime 

JDBC_URL = "jdbc:postgresql://127.0.0.1:5432/legacy_claims_db"
CONNECTION_PROPS = {
    "user": "meridian_admin",
    "password": dbutils.secrets.get(scope = "meridian-legacy-db", key = "pg-password"),
    "driver":"org.postgresql.Driver"
    
}


CATALOG = "healthcare_lakehouse"
BRONZE_SCHEMA = 'bronze'

tables = {
    "payer_master":"payers",
    "org_master":"organizations",
    "provider_master":"providers",
    "patient_master":"patients",
    "encounter_master":"encounters",
    "allergy":"allergies",
    "careplan":"careplans",
    "immunizatn":"immunizations",
    "device":"devices",
    "encountr": "encounters",
    "dx_condition":"conditions",
    "rx_med":"medications",
    "payer_xfer":"payer_transitions",
    "supply":"supplies",
    "claims_hdr":"claims",
    "img_study": "imaging_studies",
    "procedur":"procedures",
    "observatn":"observations",
    "claim_line":"claim_transactions"
}

batch_id = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

def ingest_table(source_table, target_table):
    print(f"Ingesting {source_table} to {target_table}")
    df = spark.read.jdbc(url=JDBC_URL, table=source_table, properties=CONNECTION_PROPS)

    df = (
        df
        .withColumn("_migration_batch_id", f.lit(batch_id))
        .withColumn("_legacy_source_table", f.lit(source_table))
        .withColumn("_migrated_at", f.current_timestamp())
    )

    full_target = f"{CATALOG}.{BRONZE_SCHEMA}.{target_table}"

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_target)

    count = spark.table(full_target).count()
    print(f"Inserted {count} records into {full_target}")


In [0]:
ingest_table("payer_master", "payers")


In [0]:
ingest_table("org_master", "organizations")

ingest_table("provider_master", "providers")

ingest_table("patient_master", "patients")

ingest_table("allergy", "allergies")

ingest_table("careplan", "careplans")

In [0]:
ingest_table("immunizatn", "immunizations")
ingest_table("device", "devices")
ingest_table("encountr", "encounters")
ingest_table("dx_condition", "conditions")
ingest_table("rx_med", "medications")
ingest_table("payer_xfer", "payer_transitions")
ingest_table("supply", "supplies")
ingest_table("claim_hdr", "claims")
ingest_table("img_study", "imaging_studies")

In [0]:
ingest_table("procedur", "procedures")


In [0]:
ingest_table("observatn", "observations")

In [0]:
ingest_table("claim_line", "claim_transactions")

In [0]:
# Special handling for claim_line — too large for a single unpartitioned JDBC read
bounds = spark.read.jdbc(
    url=JDBC_URL,
    table="(SELECT MIN(txn_id) as min_id, MAX(txn_id) as max_id FROM claim_line) as bounds",
    properties=CONNECTION_PROPS
).collect()[0]

min_id, max_id = bounds["min_id"], bounds["max_id"]
print(f"txn_id range: {min_id} to {max_id}")

df = (spark.read.format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", "claim_line")
    .option("user", CONNECTION_PROPS["user"])
    .option("password", CONNECTION_PROPS["password"])
    .option("driver", CONNECTION_PROPS["driver"])
    .option("partitionColumn", "txn_id")
    .option("lowerBound", min_id)
    .option("upperBound", max_id)
    .option("numPartitions", 8)
    .load()
)

df = (df
    .withColumn("_legacy_source_table", f.lit("claim_line"))
    .withColumn("_migration_batch_id", f.lit(batch_id))
    .withColumn("_migrated_at", f.current_timestamp())
)

full_target = f"{CATALOG}.{BRONZE_SCHEMA}.claims_transactions"
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_target)

count = spark.table(full_target).count()
print(f"done: {count} rows written to {full_target}")

In [0]:
expected = {
    "payers": 10,
    "organizations": 1180,
    "providers": 1157,
    "patients": 15389,
    "allergies": 14000,
    "careplans": 50616,
    "immunizations": 216742,
    "devices": 90831,
    "encounters": 943077,
    "conditions": 568478,
    "medications": 816597,
    "payer_transitions": 559188,
    "supplies": 412900,
    "claims": 1776006,
    "imaging_studies": 1482268,
    "procedures": 2569186,
    "observations": 12241431,
    "claims_transactions": 15168041,
}

print(f"{'table':<22}{'expected':>12}{'actual':>12}{'match':>8}")
for table, exp_count in expected.items():
    actual = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{table}").count()
    match = "OK" if actual == exp_count else "MISMATCH"
    print(f"{table:<22}{exp_count:>12}{actual:>12}{match:>8}")
    